In [4]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                     08_maintenance.ipynb                                   ║
# ║  Maintenance utilities — backup, cleanup, inspect, repair                  ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ── Imports ───────────────────────────────────────────────────────────
import shutil
import zipfile
import os
import numpy as np
from pathlib import Path
from datetime import datetime
from prettytable import PrettyTable

from brainvision.constants import *

RESULTS_DIR     = Path(RESULTS_DIR)
CHECKPOINTS_DIR = Path(CHECKPOINTS_DIR)
BACKUP_DIR      = Path("../backup")
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

def timestamp() -> str:
    return datetime.now().strftime("%Y%m%d_%H%M%S")

print("Maintenance notebook ready.")
print(f"  Results dir     : {RESULTS_DIR}")
print(f"  Checkpoints dir : {CHECKPOINTS_DIR}")
print(f"  Backup dir      : {BACKUP_DIR}")

Maintenance notebook ready.
  Results dir     : ../results
  Checkpoints dir : ../checkpoints
  Backup dir      : ../backup


In [2]:
# ── Inspect — what's currently on disk ────────────────────────────────
def inspect_dirs():
    """Print a summary of all files currently in results and checkpoints."""

    def _summarise(folder: Path, label: str):
        if not folder.exists():
            print(f"  {label}: folder does not exist")
            return

        files    = sorted(folder.rglob("*"))
        files    = [f for f in files if f.is_file()]
        total_mb = sum(f.stat().st_size for f in files) / 1e6

        exts = {}
        for f in files:
            exts[f.suffix] = exts.get(f.suffix, 0) + 1

        print(f"\n  {label}  ({folder})")
        print(f"  {'─'*50}")
        print(f"  Files     : {len(files)}")
        print(f"  Size      : {total_mb:.1f} MB")
        print(f"  By type   : " +
              "  ".join(f"{ext or 'no-ext'}={n}" for ext, n in sorted(exts.items())))

    print(f"\n{'═'*60}")
    print(f"  DISK INSPECTION")
    print(f"{'═'*60}")
    _summarise(RESULTS_DIR,     "Results")
    _summarise(CHECKPOINTS_DIR, "Checkpoints")
    _summarise(BACKUP_DIR,      "Backups")


inspect_dirs()


════════════════════════════════════════════════════════════
  DISK INSPECTION
════════════════════════════════════════════════════════════

  Results  (../results)
  ──────────────────────────────────────────────────
  Files     : 297
  Size      : 15.6 MB
  By type   : .npy=172  .png=125

  Checkpoints  (../checkpoints)
  ──────────────────────────────────────────────────
  Files     : 79
  Size      : 973.4 MB
  By type   : .pt=79
  Backups: folder does not exist


In [3]:
# ── Inventory table — one row per run ─────────────────────────────────
def print_inventory():
    """
    Print a table showing every run and which files exist for it.
    Columns: run_name | history | test_metrics | checkpoint | curves
    """
    if not RESULTS_DIR.exists():
        print("Results directory does not exist.")
        return

    # Collect all unique run stems from history files
    history_files = sorted(RESULTS_DIR.glob("*_history.npy"))
    if not history_files:
        print("No history files found.")
        return

    table = PrettyTable()
    table.field_names = ["Run name", "History", "Metrics", "Checkpoint",
                         "CM plot", "Curves plot"]
    table.align["Run name"] = 'l'
    for col in ["History", "Metrics", "Checkpoint", "CM plot", "Curves plot"]:
        table.align[col] = 'c'

    for h_path in history_files:
        stem     = h_path.stem.replace("_history", "")
        ckpt     = CHECKPOINTS_DIR / f"{stem}.pt"
        metrics  = RESULTS_DIR     / f"{stem}_test_metrics.npy"
        cm_plot  = RESULTS_DIR     / f"{stem}_cm.png"
        curves   = RESULTS_DIR     / f"{stem}_curves.png"

        table.add_row([
            stem,
            "✅" if h_path.exists() else "❌",
            "✅" if metrics.exists() else "❌",
            "✅" if ckpt.exists()    else "❌",
            "✅" if cm_plot.exists() else "❌",
            "✅" if curves.exists()  else "❌",
        ])

    print(f"\n{'═'*60}")
    print(f"  INVENTORY — {len(history_files)} runs found")
    print(f"{'═'*60}")
    print(table)


print_inventory()


════════════════════════════════════════════════════════════
  INVENTORY — 74 runs found
════════════════════════════════════════════════════════════
+-------------------------------------+---------+---------+------------+---------+-------------+
| Run name                            | History | Metrics | Checkpoint | CM plot | Curves plot |
+-------------------------------------+---------+---------+------------+---------+-------------+
| 1dcnn_ce_bal_fold1_vpfabelo         |    ✅   |    ✅   |     ✅     |    ✅   |      ✅     |
| 1dcnn_ce_bal_fold2_vpfabelo         |    ✅   |    ✅   |     ✅     |    ✅   |      ✅     |
| 1dcnn_ce_bal_fold3_vpfabelo         |    ✅   |    ✅   |     ✅     |    ✅   |      ✅     |
| 1dcnn_ce_bal_fold4_vpfabelo         |    ✅   |    ✅   |     ✅     |    ✅   |      ✅     |
| 1dcnn_ce_bal_fold5_vpfabelo         |    ✅   |    ✅   |     ✅     |    ✅   |      ✅     |
| 1dcnn_ce_bal_vp1                    |    ✅   |    ✅   |     ✅     |    ❌   |      ❌     |
| 1dcn

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#            DELETE ALL RESULTS AND CHECKPOINTS
#            !! BACKS UP BEFORE DELETING — ALWAYS !!
# ══════════════════════════════════════════════════════════════════════════════

# Safety flags — both must be True to execute deletion
CONFIRM_DELETE  = False   # ← set True when you are sure
EXECUTE_DELETE  = False   # ← set True only after reviewing the dry run

def backup_and_delete(dry_run: bool = True):
    """
    Back up results and checkpoints to a timestamped zip, then delete.

    Args:
        dry_run : if True, only show what would happen — no files touched
    """
    ts          = timestamp()
    backup_path = BACKUP_DIR / f"backup_{ts}.zip"

    # ── Collect all files to backup ───────────────────────────────────────────
    to_backup = []
    for folder in [RESULTS_DIR, CHECKPOINTS_DIR]:
        if folder.exists():
            for f in sorted(folder.rglob("*")):
                if f.is_file():
                    to_backup.append(f)

    if not to_backup:
        print("Nothing to backup or delete — both folders are empty.")
        return

    total_mb = sum(f.stat().st_size for f in to_backup) / 1e6

    print(f"\n{'═'*60}")
    print(f"  {'DRY RUN — ' if dry_run else ''}BACKUP + DELETE")
    print(f"{'═'*60}")
    print(f"  Files to backup : {len(to_backup)}")
    print(f"  Total size      : {total_mb:.1f} MB")
    print(f"  Backup path     : {backup_path}")
    print(f"  Folders to wipe : {RESULTS_DIR}")
    print(f"                    {CHECKPOINTS_DIR}")

    if dry_run:
        print(f"\n  Files that would be backed up and deleted:")
        for f in to_backup:
            rel = f.relative_to(f.parents[1])
            print(f"    {rel}")
        print(f"\n  ℹ️  Dry run — set EXECUTE_DELETE=True to apply")
        return

    # ── Step 1: Backup ────────────────────────────────────────────────────────
    print(f"\n  Step 1: Creating backup...")
    with zipfile.ZipFile(backup_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in to_backup:
            arcname = f.relative_to(Path("/kaggle/working"))
            zf.write(f, arcname=arcname)

    backup_mb = backup_path.stat().st_size / 1e6
    print(f"  ✅ Backup created: {backup_path.name}  ({backup_mb:.1f} MB)")

    # Verify backup is readable before deleting
    with zipfile.ZipFile(backup_path, 'r') as zf:
        bad = zf.testzip()
        if bad:
            print(f"  ❌ Backup corrupted at: {bad} — ABORTING deletion")
            return
    print(f"  ✅ Backup verified — {len(zf.namelist())} files intact")

    # ── Step 2: Delete ────────────────────────────────────────────────────────
    print(f"\n  Step 2: Deleting...")
    deleted = 0
    for folder in [RESULTS_DIR, CHECKPOINTS_DIR]:
        if folder.exists():
            shutil.rmtree(folder)
            folder.mkdir(parents=True, exist_ok=True)   # recreate empty
            print(f"  ✅ Cleared: {folder}")
            deleted += 1

    print(f"\n  ✅ Done — {len(to_backup)} files deleted")
    print(f"  ✅ Backup preserved at: {backup_path}")
    print(f"\n  To restore, run the RESTORE cell below with:")
    print(f"    RESTORE_FROM = '{backup_path}'")


# ── Safety check ──────────────────────────────────────────────────────────────
if not CONFIRM_DELETE:
    print("ℹ️  Running in dry-run mode.")
    print("   Review the output, then set CONFIRM_DELETE=True and "
          "EXECUTE_DELETE=True to apply.\n")
    backup_and_delete(dry_run=True)

elif CONFIRM_DELETE and not EXECUTE_DELETE:
    print("⚠️  CONFIRM_DELETE=True but EXECUTE_DELETE=False.")
    print("   Set EXECUTE_DELETE=True to apply the deletion.\n")
    backup_and_delete(dry_run=True)

elif CONFIRM_DELETE and EXECUTE_DELETE:
    print("⚠️  Executing backup + delete in 3 seconds...")
    import time
    time.sleep(3)
    backup_and_delete(dry_run=False)

In [ ]:
# ── Restore from a backup ────────────────────────────────────────────
RESTORE_FROM = ""   # ← paste backup path here e.g.
                    #   "/kaggle/working/backup/backup_20260714_093012.zip"
EXECUTE_RESTORE = False

def restore_backup(zip_path: str, dry_run: bool = True):
    """
    Restore results and checkpoints from a backup zip.

    Args:
        zip_path : path to the backup zip file
        dry_run  : if True, only list contents — no files extracted
    """
    zp = Path(zip_path)
    if not zp.exists():
        print(f"  ❌ Backup file not found: {zip_path}")
        return

    with zipfile.ZipFile(zp, 'r') as zf:
        names    = zf.namelist()
        total_mb = sum(i.file_size for i in zf.infolist()) / 1e6

        print(f"\n{'═'*60}")
        print(f"  {'DRY RUN — ' if dry_run else ''}RESTORE")
        print(f"{'═'*60}")
        print(f"  Backup file : {zp.name}")
        print(f"  Files       : {len(names)}")
        print(f"  Total size  : {total_mb:.1f} MB")
        print(f"  Restore to  : /kaggle/working/")

        if dry_run:
            print(f"\n  Files that would be restored:")
            for name in names[:20]:
                print(f"    {name}")
            if len(names) > 20:
                print(f"    ... and {len(names)-20} more")
            print(f"\n  ℹ️  Dry run — set EXECUTE_RESTORE=True to apply")
            return

        zf.extractall("/kaggle/working/")

    print(f"\n  ✅ Restored {len(names)} files to /kaggle/working/")


if RESTORE_FROM and EXECUTE_RESTORE:
    restore_backup(RESTORE_FROM, dry_run=False)
elif RESTORE_FROM:
    restore_backup(RESTORE_FROM, dry_run=True)
else:
    print("ℹ️  Set RESTORE_FROM to a backup zip path to restore.")

In [ ]:
# ── List all backups ──────────────────────────────────────────────────
def list_backups():
    """Print all available backups with size and file count."""
    backups = sorted(BACKUP_DIR.glob("backup_*.zip"))
    if not backups:
        print("No backups found.")
        return

    table = PrettyTable()
    table.field_names = ["Backup file", "Size (MB)", "Files", "Created"]
    table.align["Backup file"] = 'l'
    table.align["Size (MB)"]   = 'r'
    table.align["Files"]       = 'r'

    for bp in backups:
        with zipfile.ZipFile(bp, 'r') as zf:
            n_files = len(zf.namelist())
        size_mb  = bp.stat().st_size / 1e6
        created  = datetime.fromtimestamp(
            bp.stat().st_mtime).strftime("%Y-%m-%d %H:%M")
        table.add_row([bp.name, f"{size_mb:.1f}", n_files, created])

    total_mb = sum(bp.stat().st_size for bp in backups) / 1e6
    print(f"\n{'═'*60}")
    print(f"  BACKUPS  ({len(backups)} found, {total_mb:.1f} MB total)")
    print(f"{'═'*60}")
    print(table)


list_backups()

In [ ]:
# ── Delete a specific run ─────────────────────────────────────────────
DELETE_RUN_NAME  = ""     # ← e.g. "1dcnn_ce_bal_fold1_vpfabelo"
EXECUTE_RUN_DELETE = False

def delete_run(run_name: str, dry_run: bool = True):
    """
    Delete all files associated with a single run.
    Does NOT create a backup — use only for runs you are sure about.
    """
    if not run_name:
        print("ℹ️  Set DELETE_RUN_NAME to a run name to delete.")
        return

    targets = [
        RESULTS_DIR     / f"{run_name}_history.npy",
        RESULTS_DIR     / f"{run_name}_test_metrics.npy",
        RESULTS_DIR     / f"{run_name}_cm.png",
        RESULTS_DIR     / f"{run_name}_curves.png",
        CHECKPOINTS_DIR / f"{run_name}.pt",
    ]

    existing = [f for f in targets if f.exists()]
    missing  = [f for f in targets if not f.exists()]

    print(f"\n{'═'*60}")
    print(f"  {'DRY RUN — ' if dry_run else ''}DELETE RUN: {run_name}")
    print(f"{'═'*60}")
    print(f"  Files found    : {len(existing)}")
    print(f"  Files missing  : {len(missing)}")

    for f in existing:
        size_kb = f.stat().st_size / 1e3
        print(f"  {'[would delete]' if dry_run else '[deleting]'} "
              f"{f.name}  ({size_kb:.1f} KB)")
    for f in missing:
        print(f"  [not found]   {f.name}")

    if dry_run:
        print(f"\n  ℹ️  Dry run — set EXECUTE_RUN_DELETE=True to apply")
        return

    for f in existing:
        f.unlink()
    print(f"\n  ✅ Deleted {len(existing)} files for run: {run_name}")


delete_run(DELETE_RUN_NAME, dry_run=not EXECUTE_RUN_DELETE)

In [ ]:
# ── Delete old backups ────────────────────────────────────────────────
KEEP_N_BACKUPS       = 3      # keep the N most recent backups
EXECUTE_BACKUP_CLEAN = False

def clean_old_backups(keep_n: int = 3, dry_run: bool = True):
    """
    Delete all but the N most recent backup zips.

    Args:
        keep_n  : number of most recent backups to keep
        dry_run : if True, only show what would be deleted
    """
    backups = sorted(BACKUP_DIR.glob("backup_*.zip"),
                     key=lambda f: f.stat().st_mtime,
                     reverse=True)

    to_keep   = backups[:keep_n]
    to_delete = backups[keep_n:]

    print(f"\n{'═'*60}")
    print(f"  {'DRY RUN — ' if dry_run else ''}CLEAN OLD BACKUPS")
    print(f"{'═'*60}")
    print(f"  Total backups : {len(backups)}")
    print(f"  Keeping       : {len(to_keep)}")
    print(f"  Deleting      : {len(to_delete)}")

    for bp in to_keep:
        size_mb = bp.stat().st_size / 1e6
        print(f"  [keep]   {bp.name}  ({size_mb:.1f} MB)")
    for bp in to_delete:
        size_mb = bp.stat().st_size / 1e6
        print(f"  {'[would delete]' if dry_run else '[deleting]'} "
              f"{bp.name}  ({size_mb:.1f} MB)")

    if dry_run:
        print(f"\n  ℹ️  Dry run — set EXECUTE_BACKUP_CLEAN=True to apply")
        return

    freed = 0
    for bp in to_delete:
        freed += bp.stat().st_size
        bp.unlink()
    print(f"\n  ✅ Deleted {len(to_delete)} backups, "
          f"freed {freed/1e6:.1f} MB")


clean_old_backups(keep_n=KEEP_N_BACKUPS, dry_run=not EXECUTE_BACKUP_CLEAN)